In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
orders_bronze_path = f"{BRONZE_PATH}/orders"
orders_silver_path = f"{SILVER_PATH}/orders"

In [0]:
df_orders_bronze = spark.read.format("delta") \
    .load(orders_bronze_path)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
windowSpec = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())

df_orders_silver = df_orders_bronze \
    .withColumn(
        "rn",
        F.row_number().over(windowSpec)
    ) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

In [0]:
%skip
df_orders_silver.write.format("delta").mode("append").save(orders_silver_path)

In [0]:
from delta.tables import DeltaTable

orders_silver_table = DeltaTable.forPath(
    spark,
    orders_silver_path
)

orders_silver_table.alias("target") \
    .merge(
        df_orders_silver.alias("source"),
        "source.order_id = target.order_id"
    ) \
    .whenMatchedUpdate(
        set = {
            "customer_id": "source.customer_id",
            "order_date": "source.order_date",
            "order_status": "source.order_status",
            "total_amount": "source.total_amount",
            "updated_at": "source.updated_at"
        }
    ) \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
spark.read.format("delta").load(orders_silver_path).count()